# Transformer Intrinsic Optimizer for QAOA

**Learning objectives**

1. Derive the QAOA objective for MaxCut and connect it to a GPU-executable simulation workload.
2. Build a graph-conditioned Transformer that acts as an **intrinsic optimizer** for QAOA angles.
3. Train the learned optimizer by unrolling the hybrid loop over a family of graph instances.
4. Evaluate the learned policy, apply optional local refinement, and report approximation ratio.

**Framework integration.** The code path exposes a CUDA-Q integration compatibility for scalable execution and a PyTorch statevector fallback for classroom portability.

> **Where you are: Morning Block, Notebook 01 — Learned Optimization for Variational Models.**
> Building directly on the CUDA-Q kernels and GPU sampling from `00_cudaq_basics.ipynb`, this notebook builds the first complete hybrid QML workflow of the tutorial: a Transformer-based *meta-optimizer* that predicts QAOA parameter updates and generalizes across unseen graph instances, integrating CUDA-Q with PyTorch in a single end-to-end training pipeline.

## Formulation: MaxCut as a QAOA workload

For an undirected graph $G=(V,E)$ with $n=|V|$, the MaxCut value of a bitstring  
$\mathbf{z}\in\{0,1\}^n$ is

$$
C_G(\mathbf{z})
=
\sum_{(i,j)\in E}
\mathbf{1}[z_i\neq z_j].
$$

Equivalently, using the spin variable $s_i=(-1)^{z_i}\in\{-1,+1\}$, the cut objective can be written as

$$
C_G(\mathbf{s})
=
\sum_{(i,j)\in E}
\frac{1}{2}\left(1-s_i s_j\right).
$$

Using Pauli-$Z$ operators, the corresponding QAOA cost Hamiltonian is

$$
H_C
=
\sum_{(i,j)\in E}
\frac{1}{2}\left(I-Z_iZ_j\right),
$$

where $Z_i$ acts on qubit $i$. The standard transverse-field mixer Hamiltonian is

$$
H_M
=
\sum_{i=1}^{n} X_i,
$$

where $X_i$ is the Pauli-$X$ operator acting on qubit $i$.

For QAOA depth $p$, the variational state is prepared as

$$
|\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta};G)\rangle
=
\prod_{\ell=1}^{p}
e^{-i\beta_\ell H_M}
e^{-i\gamma_\ell H_C}
|+\rangle^{\otimes n}.
$$

The variational parameter vector is

$$
\boldsymbol{\theta}
=
[\gamma_1,\ldots,\gamma_p,\beta_1,\ldots,\beta_p]
\in \mathbb{R}^{2p}.
$$

The classical outer loop optimizes the expected cut value

$$
f_G(\boldsymbol{\theta})
=
\langle
\psi_p(\boldsymbol{\theta};G)
|
H_C
|
\psi_p(\boldsymbol{\theta};G)
\rangle.
$$

Thus, the QAOA parameter-optimization problem is

$$
\boldsymbol{\theta}^{\star}(G)
=
\arg\max_{\boldsymbol{\theta}\in\Omega}
f_G(\boldsymbol{\theta}),
$$

where a common parameter domain is

$$
\Omega
=
[-\pi,\pi]^p
\times
\left[-\frac{\pi}{2},\frac{\pi}{2}\right]^p.
$$

In this tutorial, we replace a hand-designed classical optimizer with a learned update rule. Instead of using a fixed optimizer such as Adam, COBYLA, SPSA, or Nelder--Mead, we learn a graph-conditioned optimizer

$$
\mathcal{F}_{\phi}
:
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
\widehat{f}_G^{(t)},
\widehat{f}_{G,\mathrm{best}}^{(t)},
t/T
\right)
\mapsto
\Delta\boldsymbol{\theta}^{(t)}.
$$

The QAOA parameters are then updated by

$$
\boldsymbol{\theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\theta}^{(t)}
+
\Delta\boldsymbol{\theta}^{(t)}
\right),
$$

where $\Pi_{\Omega}$ projects the updated parameters back into the admissible QAOA parameter domain.

The central idea is therefore to treat the classical optimizer itself as a learnable component of the hybrid quantum--classical algorithm.

## Runtime model and backend policy

The tutorial supports two simulator backends:

- **CUDA-Q / NVIDIA target**: Using CUDA-Q kernels and `cudaq.observe` / `observe_async` where available for scalable simulation on GPU.
- **PyTorch statevector fallback**: a differentiable reference implementation for small \(n\). This keeps the notebook executable in CPU-only environments.

The fallback is pedagogical rather than scalable. Statevector simulation requires memory \(O(2^n)\), while the hybrid training loop repeatedly evaluates the QAOA objective over many graph instances. In an SC tutorial, this naturally motivates GPU acceleration and batched execution.

## Imports and optional CUDA-Q detection

In [1]:
import copy
import math
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn

# Optional CUDA-Q path. The notebook remains executable with the PyTorch fallback.
try:
    import cudaq
    from cudaq import spin
except ImportError:
    cudaq = None
    spin = None

## Configuration

The configuration cell centralizes all experimental controls: graph size, QAOA depth, meta-training horizon, Transformer size, and backend selection.

The default `SC_TUTORIAL_FAST_MODE=True` is chosen for a live hands-on tutorial. Set it to `False` to restore the heavier original demo values: 48 training graphs, 12 validation graphs, 200 meta-epochs, an 8-step unroll, and a wider Transformer.

In [2]:
# Fast defaults are selected for an interactive SC tutorial session.
# Set SC_TUTORIAL_FAST_MODE = False to restore the original heavier demo settings.
SC_TUTORIAL_FAST_MODE = True

@dataclass
class Config:
    n_qubits: int = 5
    p: int = 2

    train_graph_count: int = 16 if SC_TUTORIAL_FAST_MODE else 48
    val_graph_count: int = 4 if SC_TUTORIAL_FAST_MODE else 12
    batch_size: int = 2 if SC_TUTORIAL_FAST_MODE else 12

    meta_epochs: int = 5 if SC_TUTORIAL_FAST_MODE else 200
    unroll_steps: int = 3 if SC_TUTORIAL_FAST_MODE else 8
    lr: float = 2e-3

    d_model: int = 32 if SC_TUTORIAL_FAST_MODE else 160
    nhead: int = 4
    num_layers: int = 1 if SC_TUTORIAL_FAST_MODE else 2
    dim_feedforward: int = 64 if SC_TUTORIAL_FAST_MODE else 320
    dropout: float = 0.0
    weight_decay: float = 1e-5

    init_angle_noise: float = 0.20

    refine_steps: int = 5 if SC_TUTORIAL_FAST_MODE else 40
    refine_lr: float = 5e-2

    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: torch.dtype = torch.float32
    complex_dtype: torch.dtype = torch.complex64

    qc_simulator_backend: str = "cudaq" if cudaq is not None else "torch"  # "cudaq" or "torch"
    cudaq_target: str = "nvidia"
    cudaq_target_option: str | None = "fp32"
    cudaq_grad_epsilon: float = 1e-4
    cudaq_kernel_mode: str = "jit"  # "jit" or "builder"
    cudaq_use_async_forward: bool = True

    # Stop meta-training if validation hasn't improved for this many checks.
    # Set to 0 to disable.
    early_stop_patience: int = 5


CFG = Config()

if CFG.qc_simulator_backend == "torch" and cudaq is None:
    print(
        "CUDA-Q is not installed; using the PyTorch statevector backend. "
        "For NVIDIA GPU acceleration in the SC tutorial environment, install CUDA-Q "
        "and set CFG.qc_simulator_backend='cudaq'."
    )

## Reproducibility and cached graph metadata

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)

In [4]:
# Keyed by id() of the on-device adjacency tensor. precompute_graph_info()
# moves graphs to the target device once, then everything downstream looks up
# the same id().
_GRAPH_INFO_CACHE: Dict[int, Dict[str, Any]] = {}


def _graph_info(adj: torch.Tensor) -> Dict[str, Any] | None:
    return _GRAPH_INFO_CACHE.get(id(adj))

## Graph utilities

Each MaxCut instance is represented by a symmetric adjacency matrix

$$
A \in \mathbb{R}^{n\times n},
\qquad
A_{ij}=A_{ji},
\qquad
A_{ii}=0.
$$

The learned optimizer receives a fixed-size graph feature vector

$$
\mathbf{x}_G
=
\left[
\operatorname{vec}_{\triangle}(A),
\operatorname{density}(G),
\operatorname{degree\_profile}(G)
\right].
$$

For fixed graph size $n$, $\operatorname{vec}_{\triangle}(A)$ denotes the upper-triangular vectorization of the adjacency matrix:

$$
\operatorname{vec}_{\triangle}(A)
=
\left[
A_{12},A_{13},\ldots,A_{1n},
A_{23},\ldots,A_{2n},
\ldots,
A_{n-1,n}
\right]
\in
\mathbb{R}^{n(n-1)/2}.
$$

The graph density is defined as

$$
\operatorname{density}(G)
=
\frac{2|E|}{n(n-1)}.
$$

The degree of node $i$ is

$$
d_i
=
\sum_{j=1}^{n} A_{ij}.
$$

A compact degree-profile vector can be constructed using summary statistics such as

$$
\operatorname{degree\_profile}(G)
=
\left[
\frac{1}{n}\sum_{i=1}^{n} d_i,
\operatorname{std}(d_1,\ldots,d_n),
\min_i d_i,
\max_i d_i
\right].
$$

Therefore, a simple fixed-size graph representation is

$$
\mathbf{x}_G
=
\left[
\operatorname{vec}_{\triangle}(A),
\frac{2|E|}{n(n-1)},
\frac{1}{n}\sum_{i=1}^{n} d_i,
\operatorname{std}(d_1,\ldots,d_n),
\min_i d_i,
\max_i d_i
\right].
$$

This representation is sufficient for a compact tutorial implementation because all graphs have the same number of vertices. For larger graphs or variable-size instances, this block can be replaced by a graph neural network, message-passing network, or graph Transformer encoder.

In [5]:
def make_adj(n: int, edges: List[Tuple[int, int]], device=None, dtype=None) -> torch.Tensor:
    adj = torch.zeros((n, n), dtype=dtype or CFG.dtype, device=device)
    for i, j in edges:
        adj[i, j] = 1.0
        adj[j, i] = 1.0
    return adj


def _compute_edge_list(adj: torch.Tensor) -> List[Tuple[int, int]]:
    n = adj.shape[0]
    iu = torch.triu_indices(n, n, offset=1, device=adj.device)
    mask = adj[iu[0], iu[1]] > 0.5
    src = iu[0][mask].tolist()
    tgt = iu[1][mask].tolist()
    return list(zip(src, tgt))


def edge_list_from_adj(adj: torch.Tensor) -> List[Tuple[int, int]]:
    info = _graph_info(adj)
    if info is not None:
        return list(info["edges"])
    return _compute_edge_list(adj)


def graph_key(adj: torch.Tensor) -> Tuple[int, ...]:
    n = adj.shape[0]
    iu = torch.triu_indices(n, n, offset=1, device=adj.device)
    return tuple(int(v) for v in adj[iu[0], iu[1]].tolist())


def _compute_upper_triangular_features(adj: torch.Tensor) -> torch.Tensor:
    n = adj.shape[0]
    iu = torch.triu_indices(n, n, offset=1, device=adj.device)
    feat = adj[iu[0], iu[1]]
    density = feat.mean().unsqueeze(0)
    degree_profile = adj.sum(dim=1) / max(1, n - 1)
    return torch.cat([feat, density, degree_profile], dim=0)


def upper_triangular_features(adj: torch.Tensor) -> torch.Tensor:
    info = _graph_info(adj)
    if info is not None:
        return info["features"]
    return _compute_upper_triangular_features(adj)


def random_connected_graph(n: int, p_edge: float = 0.5) -> torch.Tensor:
    while True:
        adj = torch.zeros((n, n), dtype=CFG.dtype)
        for i in range(n):
            for j in range(i + 1, n):
                if random.random() < p_edge:
                    adj[i, j] = 1.0
                    adj[j, i] = 1.0
        if len(_compute_edge_list(adj)) == 0:
            continue
        if is_connected(adj):
            return adj


def is_connected(adj: torch.Tensor) -> bool:
    n = adj.shape[0]
    visited = [False] * n
    stack = [0]
    visited[0] = True
    while stack:
        u = stack.pop()
        for v in range(n):
            if adj[u, v] > 0.5 and not visited[v]:
                visited[v] = True
                stack.append(v)
    return all(visited)


def build_unique_graphs(num_graphs: int, n_qubits: int, excluded_keys=None) -> List[torch.Tensor]:
    graphs = []
    excluded_keys = set() if excluded_keys is None else set(excluded_keys)
    seen = set(excluded_keys)

    while len(graphs) < num_graphs:
        adj = random_connected_graph(n_qubits, p_edge=0.5)
        key = graph_key(adj)
        if key not in seen:
            seen.add(key)
            graphs.append(adj)

    return graphs

## Exact MaxCut objective and graph precomputation

For the small tutorial setting, the exact MaxCut value is computed by brute force over all bitstrings:

$$
C_G^\star
=
\max_{\mathbf{z}\in\{0,1\}^n}
C_G(\mathbf{z}).
$$

For a bitstring $\mathbf{z}=(z_1,\ldots,z_n)$, the MaxCut objective is

$$
C_G(\mathbf{z})
=
\sum_{(i,j)\in E}
\mathbf{1}[z_i\neq z_j].
$$

Equivalently, using spin variables $s_i=(-1)^{z_i}$,

$$
C_G(\mathbf{s})
=
\sum_{(i,j)\in E}
\frac{1}{2}
\left(
1-s_i s_j
\right).
$$

The exact optimum is therefore

$$
C_G^\star
=
\max_{\mathbf{s}\in\{-1,+1\}^n}
\sum_{(i,j)\in E}
\frac{1}{2}
\left(
1-s_i s_j
\right).
$$

This enables the approximation-ratio metric

$$
\rho_G(\boldsymbol{\theta})
=
\frac{
f_G(\boldsymbol{\theta})
}{
C_G^\star
},
$$

where

$$
f_G(\boldsymbol{\theta})
=
\langle
\psi_p(\boldsymbol{\theta};G)
|
H_C
|
\psi_p(\boldsymbol{\theta};G)
\rangle
$$

is the expected QAOA cut value.

In the notebook implementation, each graph is precomputed and stored in a cache containing

$$
\mathcal{C}(G)
=
\left\{
E,\,
\mathbf{x}_G,\,
\operatorname{diag}(H_C),\,
C_G^\star
\right\}.
$$

Here, $E$ is the edge list, $\mathbf{x}_G$ is the graph feature vector, $\operatorname{diag}(H_C)$ is the cost Hamiltonian diagonal in the computational basis, and $C_G^\star$ is the exact MaxCut value.

Caching these quantities avoids recomputing graph-invariant information during repeated training iterations. This is especially useful in a tutorial setting, where the same graph instances are evaluated many times under different QAOA parameters.

In [6]:
def bitstring_from_index(index: int, n: int) -> str:
    bits = [(index >> i) & 1 for i in range(n)]
    return "".join(str(b) for b in bits)


def _compute_cost_diag(adj: torch.Tensor) -> torch.Tensor:
    n = adj.shape[0]
    dim = 1 << n
    device = adj.device
    indices = torch.arange(dim, device=device)
    qubits = torch.arange(n, device=device)
    bits = ((indices.unsqueeze(1) >> qubits.unsqueeze(0)) & 1).to(adj.dtype)
    diff = (bits.unsqueeze(2) != bits.unsqueeze(1)).to(adj.dtype)
    # Each unordered edge counted twice in the dim x n x n product; divide by 2.
    return (diff * adj.unsqueeze(0)).sum(dim=(1, 2)) / 2.0


def cost_diag_from_adj(adj: torch.Tensor) -> torch.Tensor:
    info = _graph_info(adj)
    if info is not None:
        return info["cost_diag"]
    return _compute_cost_diag(adj)


def maxcut_bruteforce(adj: torch.Tensor) -> Tuple[float, str]:
    info = _graph_info(adj)
    if info is not None:
        diag = info["cost_diag"]
        _, best_index = torch.max(diag, dim=0)
        return float(info["max_cut"]), bitstring_from_index(int(best_index.item()), adj.shape[0])
    diag = _compute_cost_diag(adj)
    best_value, best_index = torch.max(diag, dim=0)
    return float(best_value.item()), bitstring_from_index(int(best_index.item()), adj.shape[0])

In [7]:
def precompute_graph_info(graphs: List[torch.Tensor], device: str) -> List[torch.Tensor]:
    """Move each graph to ``device`` and cache invariants (edges, features,
    cost diagonal, exact maxcut). Returns the on-device tensors. Subsequent
    .to(device) calls on these tensors are identity, so id() stays stable.
    """
    out: List[torch.Tensor] = []
    for g in graphs:
        g_dev = g.to(device)
        out.append(g_dev)
        if id(g_dev) in _GRAPH_INFO_CACHE:
            continue
        edges = tuple(_compute_edge_list(g_dev))
        features = _compute_upper_triangular_features(g_dev)
        cost_diag = _compute_cost_diag(g_dev)
        max_cut = float(cost_diag.max().item())
        _GRAPH_INFO_CACHE[id(g_dev)] = {
            "edges": edges,
            "features": features,
            "cost_diag": cost_diag,
            "max_cut": max_cut,
        }
    return out

## CUDA-Q programming model for quantum kernels

The CUDA-Q builds the QAOA circuit as an executable quantum kernel and evaluates the Hamiltonian expectation value. In the minibatch training loop, graph instances are evaluated through a batched wrapper. When `cudaq.observe_async` is available.

Conceptually, the CUDA-Q evaluates

$$
\widehat{f}_G(\boldsymbol{\theta})
\approx
\operatorname{observe}
\left(
U_p(\boldsymbol{\theta};G),
H_C
\right),
$$

where the QAOA unitary is

$$
U_p(\boldsymbol{\theta};G)
=
\prod_{\ell=1}^{p}
e^{-i\beta_\ell H_M}
e^{-i\gamma_\ell H_C}.
$$

The corresponding QAOA state is

$$
|\psi_p(\boldsymbol{\theta};G)\rangle
=
U_p(\boldsymbol{\theta};G)
|+\rangle^{\otimes n}.
$$

Therefore, the exact expectation value is

$$
f_G(\boldsymbol{\theta})
=
\langle
+|^{\otimes n}
U_p^\dagger(\boldsymbol{\theta};G)
H_C
U_p(\boldsymbol{\theta};G)
|+\rangle^{\otimes n}.
$$

Equivalently,

$$
f_G(\boldsymbol{\theta})
=
\langle
\psi_p(\boldsymbol{\theta};G)
|
H_C
|
\psi_p(\boldsymbol{\theta};G)
\rangle.
$$

In a shot-based execution mode, the expectation is estimated from measurement samples:

$$
\widehat{f}_G(\boldsymbol{\theta})
=
\frac{1}{S}
\sum_{s=1}^{S}
C_G\!\left(\mathbf{z}^{(s)}\right),
\qquad
\mathbf{z}^{(s)}
\sim
\left|
\langle \mathbf{z}|\psi_p(\boldsymbol{\theta};G)\rangle
\right|^2.
$$

The custom autograd wrapper estimates gradients by central finite differences for the CUDA-Q expectation path. For parameter component $\theta_k$, the finite-difference estimator is

$$
\frac{\partial \widehat{f}_G}{\partial \theta_k}
\approx
\frac{
\widehat{f}_G(\boldsymbol{\theta}+\epsilon \mathbf{e}_k)
-
\widehat{f}_G(\boldsymbol{\theta}-\epsilon \mathbf{e}_k)
}{
2\epsilon
},
$$

where $\epsilon>0$ is a small finite-difference step and $\mathbf{e}_k$ is the $k$-th coordinate basis vector.

This backend is designed to expose the quantum-expectation path cleanly while keeping the learned optimizer in PyTorch. In an SC tutorial setting, this separation makes the workflow clear: CUDA-Q handles quantum circuit execution, while PyTorch handles the Transformer-based learned update rule.

In [8]:
_CUDAQ_PROGRAM_CACHE: Dict[tuple[Any, ...], Tuple[Any, Any, Tuple[Any, ...]]] = {}
_CUDAQ_TARGET_KEY: Tuple[str, str | None] | None = None



_CUDAQ_QAOA_JIT_KERNEL = None

if cudaq is not None:

    @cudaq.kernel(defer_compilation=False)

    def _cudaq_qaoa_jit_kernel(

        n_qubits: int,

        p: int,

        edges_src: List[int],

        edges_tgt: List[int],

        theta: List[float],

    ):

        q = cudaq.qvector(n_qubits)

        h(q)

        for layer in range(p):

            for edge_idx in range(len(edges_src)):

                i = edges_src[edge_idx]

                j = edges_tgt[edge_idx]

                x.ctrl(q[i], q[j])

                rz(-theta[layer], q[j])

                x.ctrl(q[i], q[j])

            for qubit in range(n_qubits):

                rx(2.0 * theta[p + layer], q[qubit])



    _CUDAQ_QAOA_JIT_KERNEL = _cudaq_qaoa_jit_kernel


def _configure_cudaq_target() -> None:
    global _CUDAQ_TARGET_KEY
    if cudaq is None:
        raise RuntimeError(
            "CFG.qc_simulator_backend='cudaq' requires the cuda-quantum-cu13 "
            "package. Install it with: pip install cuda-quantum-cu13"
        )
    target_key = (CFG.cudaq_target, CFG.cudaq_target_option)
    if _CUDAQ_TARGET_KEY == target_key:
        return
    if _CUDAQ_TARGET_KEY is not None:
        cudaq.reset_target()
    if CFG.cudaq_target_option is None:
        cudaq.set_target(CFG.cudaq_target)
    else:
        cudaq.set_target(CFG.cudaq_target, option=CFG.cudaq_target_option)
    cudaq.set_random_seed(CFG.seed)
    _CUDAQ_TARGET_KEY = target_key


def _cudaq_edges(adj: torch.Tensor) -> Tuple[Tuple[int, int], ...]:
    info = _graph_info(adj)
    if info is not None:
        return info["edges"]
    return tuple(_compute_edge_list(adj.detach().to("cpu")))


def _cudaq_maxcut_hamiltonian(edges: Tuple[Tuple[int, int], ...]) -> Any:
    if not edges:
        return 0.0 * spin.z(0)
    hamiltonian = None
    for i, j in edges:
        term = 0.5 - 0.5 * spin.z(i) * spin.z(j)
        hamiltonian = term if hamiltonian is None else hamiltonian + term
    return hamiltonian


def _cudaq_qaoa_program(

    n_qubits: int,

    p: int,

    edges: Tuple[Tuple[int, int], ...],

) -> Tuple[Any, Any, Tuple[Any, ...]]:

    _configure_cudaq_target()

    mode = CFG.cudaq_kernel_mode

    if mode not in {"jit", "builder"}:

        raise ValueError(f"Unknown cudaq_kernel_mode: {mode}")

    cache_key = (mode, n_qubits, p, edges)

    cached = _CUDAQ_PROGRAM_CACHE.get(cache_key)

    if cached is not None:

        return cached



    if mode == "jit":

        if _CUDAQ_QAOA_JIT_KERNEL is None:

            raise RuntimeError("CUDA-Q JIT kernel is unavailable because cudaq failed to import")

        edges_src = [i for i, _ in edges]

        edges_tgt = [j for _, j in edges]

        program = (

            _CUDAQ_QAOA_JIT_KERNEL,

            _cudaq_maxcut_hamiltonian(edges),

            (n_qubits, p, edges_src, edges_tgt),

        )

    else:

        kernel, theta = cudaq.make_kernel(list)

        q = kernel.qalloc(n_qubits)

        kernel.h(q)

        for layer in range(p):

            for i, j in edges:

                kernel.cx(q[i], q[j])

                kernel.rz(-theta[layer], q[j])

                kernel.cx(q[i], q[j])

            for qubit in range(n_qubits):

                kernel.rx(2.0 * theta[p + layer], q[qubit])

        program = (kernel, _cudaq_maxcut_hamiltonian(edges), ())



    _CUDAQ_PROGRAM_CACHE[cache_key] = program

    return program





def _cudaq_expectation_value(

    n_qubits: int,

    p: int,

    edges: Tuple[Tuple[int, int], ...],

    theta_values: List[float],

) -> float:

    if not edges:

        return 0.0

    kernel, hamiltonian, static_args = _cudaq_qaoa_program(n_qubits, p, edges)

    return float(cudaq.observe(kernel, hamiltonian, *static_args, theta_values).expectation())





def _cudaq_expectation_values(

    n_qubits: int,

    p: int,

    edges: Tuple[Tuple[int, int], ...],

    theta_values_batch: List[List[float]],

) -> List[float]:

    if not edges:

        return [0.0 for _ in theta_values_batch]

    kernel, hamiltonian, static_args = _cudaq_qaoa_program(n_qubits, p, edges)

    params = np.asarray(theta_values_batch, dtype=np.float64)

    if static_args:

        static_n_qubits, static_p, edges_src, edges_tgt = static_args

        batch_size = len(theta_values_batch)

        results = cudaq.observe(

            kernel,

            hamiltonian,

            [static_n_qubits for _ in range(batch_size)],

            [static_p for _ in range(batch_size)],

            [edges_src for _ in range(batch_size)],

            [edges_tgt for _ in range(batch_size)],

            params,

        )

    else:

        results = cudaq.observe(kernel, hamiltonian, params)

    return [float(result.expectation()) for result in results]


def _cudaq_observe_per_graph(
    n_qubits: int,
    p: int,
    edges_per_graph: Tuple[Tuple[Tuple[int, int], ...], ...],
    theta_lists: List[List[float]],
) -> List[float]:
    """Forward across multiple graphs. Uses cudaq.observe_async fan-out when
    available so per-call Python dispatch is paid concurrently instead of
    serially.
    """
    observe_async = getattr(cudaq, "observe_async", None) if cudaq is not None else None
    if observe_async is None or not CFG.cudaq_use_async_forward:
        return [
            _cudaq_expectation_value(n_qubits, p, edges, tv)
            for edges, tv in zip(edges_per_graph, theta_lists)
        ]

    futures: List[Any] = []
    for edges, theta_values in zip(edges_per_graph, theta_lists):
        if not edges:
            futures.append(None)
            continue
        kernel, hamiltonian, static_args = _cudaq_qaoa_program(n_qubits, p, edges)
        try:
            f = observe_async(kernel, hamiltonian, *static_args, theta_values)
        except Exception:
            # Fall back to sync if observe_async signature differs in this build.
            return [
                _cudaq_expectation_value(n_qubits, p, edges_, tv)
                for edges_, tv in zip(edges_per_graph, theta_lists)
            ]
        futures.append(f)
    return [0.0 if f is None else float(f.get().expectation()) for f in futures]


class _CudaQQAOAExpectation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, theta: torch.Tensor, adj: torch.Tensor, p: int) -> torch.Tensor:
        edges = _cudaq_edges(adj)
        theta_values = theta.detach().to(dtype=torch.float64).cpu().tolist()
        ctx.n_qubits = int(adj.shape[0])
        ctx.p = int(p)
        ctx.edges = edges
        ctx.theta_values = theta_values
        ctx.epsilon = float(CFG.cudaq_grad_epsilon)
        value = _cudaq_expectation_value(ctx.n_qubits, ctx.p, edges, theta_values)
        return theta.new_tensor(value)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None, None]:
        epsilon = ctx.epsilon
        shifted_thetas = []
        for idx in range(len(ctx.theta_values)):
            theta_plus = list(ctx.theta_values)
            theta_minus = list(ctx.theta_values)
            theta_plus[idx] += epsilon
            theta_minus[idx] -= epsilon
            shifted_thetas.extend([theta_plus, theta_minus])

        shifted_values = _cudaq_expectation_values(ctx.n_qubits, ctx.p, ctx.edges, shifted_thetas)
        grad_values = [
            (shifted_values[2 * idx] - shifted_values[2 * idx + 1]) / (2.0 * epsilon)
            for idx in range(len(ctx.theta_values))
        ]
        grad = torch.tensor(grad_values, dtype=torch.float64, device=grad_output.device)
        return grad_output.to(grad.dtype) * grad.to(grad_output.device), None, None


class _CudaQQAOABatchedExpectation(torch.autograd.Function):
    """Batched custom autograd op covering a whole minibatch of graphs in a
    single forward/backward pair. Forward fans out across graphs via
    cudaq.observe_async (when available). Backward keeps the per-graph
    parameter-shift batched into a single cudaq.observe call (as before).
    """

    @staticmethod
    def forward(
        ctx,
        theta_batch: torch.Tensor,
        n_qubits: int,
        p: int,
        edges_per_graph: Tuple[Tuple[Tuple[int, int], ...], ...],
    ) -> torch.Tensor:
        theta_lists = theta_batch.detach().to(dtype=torch.float64).cpu().tolist()
        values = _cudaq_observe_per_graph(n_qubits, p, edges_per_graph, theta_lists)

        ctx.n_qubits = int(n_qubits)
        ctx.p = int(p)
        ctx.edges_per_graph = edges_per_graph
        ctx.theta_lists = theta_lists
        ctx.epsilon = float(CFG.cudaq_grad_epsilon)

        return theta_batch.new_tensor(values)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> Tuple[torch.Tensor, None, None, None]:
        epsilon = ctx.epsilon
        n_params = 2 * ctx.p
        grad_per_graph: List[List[float]] = []
        for b, edges in enumerate(ctx.edges_per_graph):
            theta_values = ctx.theta_lists[b]
            if not edges:
                grad_per_graph.append([0.0] * n_params)
                continue
            shifted = []
            for idx in range(n_params):
                theta_plus = list(theta_values)
                theta_minus = list(theta_values)
                theta_plus[idx] += epsilon
                theta_minus[idx] -= epsilon
                shifted.extend([theta_plus, theta_minus])
            shifted_values = _cudaq_expectation_values(ctx.n_qubits, ctx.p, edges, shifted)
            grad_b = [
                (shifted_values[2 * idx] - shifted_values[2 * idx + 1]) / (2.0 * epsilon)
                for idx in range(n_params)
            ]
            grad_per_graph.append(grad_b)

        grad = torch.tensor(grad_per_graph, dtype=torch.float64, device=grad_output.device)
        grad = grad.to(grad_output.dtype)
        return grad * grad_output.unsqueeze(-1), None, None, None


def qaoa_expectation_cudaq(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> torch.Tensor:
    theta = torch.cat([gammas, betas], dim=0)
    p = int(gammas.shape[0])
    if theta.requires_grad:
        return _CudaQQAOAExpectation.apply(theta, adj.detach(), p)

    edges = _cudaq_edges(adj)
    theta_values = theta.detach().to(device="cpu", dtype=torch.float64).tolist()
    value = _cudaq_expectation_value(int(adj.shape[0]), p, edges, theta_values)
    return theta.new_tensor(value)

## Differentiable PyTorch statevector backend

The PyTorch backend is the compact pedagogical implementation used for small-scale tutorial experiments. It explicitly constructs the full quantum statevector and applies the QAOA cost and mixer evolutions using differentiable tensor operations.

For an $n$-qubit system, the statevector is

$$
|\psi\rangle
\in
\mathbb{C}^{2^n}.
$$

The initial QAOA state is the uniform superposition

$$
|\psi^{(0)}\rangle
=
|+\rangle^{\otimes n}
=
\frac{1}{\sqrt{2^n}}
\sum_{\mathbf{z}\in\{0,1\}^n}
|\mathbf{z}\rangle.
$$

For each QAOA layer $\ell=1,\ldots,p$, the state is updated by first applying the cost evolution,

$$
|\psi\rangle
\leftarrow
e^{-i\gamma_\ell H_C}
|\psi\rangle,
$$

followed by the mixer evolution,

$$
|\psi\rangle
\leftarrow
e^{-i\beta_\ell H_M}
|\psi\rangle.
$$

Equivalently, after $p$ layers,

$$
|\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta};G)\rangle
=
\prod_{\ell=1}^{p}
e^{-i\beta_\ell H_M}
e^{-i\gamma_\ell H_C}
|+\rangle^{\otimes n}.
$$

Because the MaxCut Hamiltonian $H_C$ is diagonal in the computational basis, the cost evolution can be implemented as an elementwise phase multiplication:

$$
\psi_{\mathbf{z}}
\leftarrow
\exp\!\left(
-i\gamma_\ell C_G(\mathbf{z})
\right)
\psi_{\mathbf{z}},
$$

where $\psi_{\mathbf{z}}$ is the amplitude of basis state $|\mathbf{z}\rangle$.

The mixer Hamiltonian is

$$
H_M
=
\sum_{i=1}^{n} X_i.
$$

Since all $X_i$ terms commute, the mixer unitary factorizes as

$$
e^{-i\beta_\ell H_M}
=
\prod_{i=1}^{n}
e^{-i\beta_\ell X_i}.
$$

Each single-qubit mixer rotation is

$$
e^{-i\beta_\ell X}
=
\cos(\beta_\ell) I
-
i\sin(\beta_\ell) X.
$$

The expected cut value is computed from the final state as

$$
f_G(\boldsymbol{\theta})
=
\langle
\psi_p(\boldsymbol{\theta};G)
|
H_C
|
\psi_p(\boldsymbol{\theta};G)
\rangle.
$$

Since $H_C$ is diagonal, this becomes

$$
f_G(\boldsymbol{\theta})
=
\sum_{\mathbf{z}\in\{0,1\}^n}
\left|
\psi_{\mathbf{z}}
\right|^2
C_G(\mathbf{z}).
$$

This backend is fully differentiable in PyTorch, so gradients can be propagated through the statevector simulation and through the learned Transformer optimizer. Its main limitation is the exponential memory and runtime scaling:

$$
\dim(|\psi\rangle)=2^n.
$$

In [9]:
def rx(beta: torch.Tensor) -> torch.Tensor:
    c = torch.cos(beta)
    s = torch.sin(beta)
    row1 = torch.stack([c.to(CFG.complex_dtype), (-1j * s).to(CFG.complex_dtype)])
    row2 = torch.stack([(-1j * s).to(CFG.complex_dtype), c.to(CFG.complex_dtype)])
    return torch.stack([row1, row2], dim=0)


def kron_n(mats: List[torch.Tensor]) -> torch.Tensor:
    out = mats[0]
    for m in mats[1:]:
        out = torch.kron(out, m)
    return out


def mixer_unitary(beta: torch.Tensor, n: int) -> torch.Tensor:
    return kron_n([rx(beta) for _ in range(n)])


def qaoa_state_torch(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    n = adj.shape[0]
    dim = 1 << n
    device = adj.device

    diag = cost_diag_from_adj(adj).to(device)
    diag_c = diag.to(CFG.complex_dtype)

    state = torch.ones(dim, dtype=CFG.complex_dtype, device=device) / math.sqrt(dim)

    for gamma, beta in zip(gammas, betas):
        state = torch.exp(-1j * gamma.to(CFG.complex_dtype) * diag_c) * state
        U_mix = mixer_unitary(beta, n).to(device)
        state = U_mix @ state

    return state, diag


def qaoa_expectation_torch(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> torch.Tensor:
    state, diag = qaoa_state_torch(adj, gammas, betas)
    probs = (state.conj() * state).real
    return torch.sum(probs * diag.to(probs.device))

## Unified quantum-objective interface

In [10]:
def qaoa_expectation(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> torch.Tensor:
    if CFG.qc_simulator_backend == "cudaq":
        return qaoa_expectation_cudaq(adj, gammas, betas)
    if CFG.qc_simulator_backend != "torch":
        raise ValueError(f"Unknown qc_simulator_backend: {CFG.qc_simulator_backend}")
    return qaoa_expectation_torch(adj, gammas, betas)


def qaoa_state(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    return qaoa_state_torch(adj, gammas, betas)


def _cudaq_most_likely_bitstring(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> Tuple[str, float]:

    p = int(gammas.shape[0])

    theta = torch.cat([gammas, betas], dim=0)

    edges = _cudaq_edges(adj)

    kernel, _, static_args = _cudaq_qaoa_program(int(adj.shape[0]), p, edges)

    theta_values = theta.detach().to(device="cpu", dtype=torch.float64).tolist()

    state = cudaq.get_state(kernel, *static_args, theta_values)

    n = int(adj.shape[0])

    bitstrings = [bitstring_from_index(idx, n) for idx in range(1 << n)]

    amplitudes = state.amplitudes(bitstrings)

    best_idx = max(range(len(amplitudes)), key=lambda idx: abs(amplitudes[idx]) ** 2)

    bits = bitstrings[best_idx]

    cut = 0.0

    for i, j in edges:

        cut += 1.0 if bits[i] != bits[j] else 0.0

    return bits, cut





def most_likely_bitstring(adj: torch.Tensor, gammas: torch.Tensor, betas: torch.Tensor) -> Tuple[str, float]:
    if CFG.qc_simulator_backend == "cudaq":
        return _cudaq_most_likely_bitstring(adj, gammas, betas)
    state, diag = qaoa_state_torch(adj, gammas, betas)
    probs = (state.conj() * state).real
    idx = int(torch.argmax(probs).item())
    return bitstring_from_index(idx, adj.shape[0]), float(diag[idx].item())

## Transformer as an intrinsic optimizer

At learned-optimization step $t$, the model receives graph features, the current QAOA angles, the previous parameter update, and objective-history features. The goal is to replace a hand-designed optimizer step with a learned update rule.

For each QAOA layer $\ell\in\{1,\ldots,p\}$, define the layer token

$$
\mathbf{x}_{\ell}^{(t)}
=
\left[
\gamma_\ell^{(t)},
\beta_\ell^{(t)},
\Delta\gamma_\ell^{(t-1)},
\Delta\beta_\ell^{(t-1)},
E_t,
E_t^{\mathrm{best}},
E_t^{\mathrm{best}}-E_t,
t/T,
1-t/T
\right]^\top .
$$

Here, $\gamma_\ell^{(t)}$ and $\beta_\ell^{(t)}$ are the current QAOA angles at layer $\ell$, while $\Delta\gamma_\ell^{(t-1)}$ and $\Delta\beta_\ell^{(t-1)}$ are the previous learned updates. The scalar $E_t$ denotes the current objective value, and $E_t^{\mathrm{best}}$ denotes the best objective value observed so far:

$$
E_t
=
\widehat{f}_G(\boldsymbol{\theta}^{(t)}),
\qquad
E_t^{\mathrm{best}}
=
\max_{0\leq \tau\leq t}
\widehat{f}_G(\boldsymbol{\theta}^{(\tau)}).
$$

The quantity

$$
E_t^{\mathrm{best}}-E_t
$$

measures the current optimality gap relative to the best value found along the trajectory. The normalized time features

$$
\frac{t}{T},
\qquad
1-\frac{t}{T}
$$

allow the optimizer to distinguish early exploration from later refinement.

The graph feature vector is embedded into a graph token

$$
\mathbf{g}
=
\phi_G(G),
$$

where $\phi_G$ is a graph encoder. In the compact tutorial implementation, $\phi_G$ may be a multilayer perceptron applied to fixed-size graph features. For larger or variable-size graphs, $\phi_G$ can be replaced by a graph neural network or graph Transformer.

The graph token and the $p$ QAOA-layer tokens are then processed jointly by a Transformer encoder:

$$
[
\mathbf{h}_G,
\mathbf{h}_1^{(t)},
\ldots,
\mathbf{h}_p^{(t)}
]
=
\operatorname{Transformer}_{\phi}
\left(
[
\mathbf{g},
\mathbf{x}_1^{(t)},
\ldots,
\mathbf{x}_p^{(t)}
]
\right).
$$

The self-attention mechanism allows the optimizer to model dependencies between graph structure, QAOA layers, current angles, previous updates, and optimization history. This is useful because QAOA parameters are not independent: the update of one layer can depend on the behavior of other layers and on the structure of the input graph.

The Transformer output corresponding to the QAOA-layer tokens is passed through a prediction head:

$$
\mathbf{r}^{(t)}
=
\operatorname{Head}_{\phi}
\left(
\mathbf{h}_{1:p}^{(t)}
\right)
\in
\mathbb{R}^{2p}.
$$

A bounded update is then produced by

$$
\Delta\boldsymbol{\theta}^{(t)}
=
\alpha
\tanh
\left(
\mathbf{r}^{(t)}
\right),
$$

where $\alpha>0$ controls the maximum update magnitude. Equivalently,

$$
\Delta\boldsymbol{\theta}^{(t)}
=
\alpha
\tanh
\left(
\operatorname{Head}_{\phi}
\left(
\mathbf{h}_{1:p}^{(t)}
\right)
\right).
$$

The QAOA parameters are updated as

$$
\boldsymbol{\theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\theta}^{(t)}
+
\Delta\boldsymbol{\theta}^{(t)}
\right),
$$

where $\Pi_{\Omega}$ projects the parameters back to the admissible QAOA domain.

Thus, the Transformer implements an intrinsic optimizer:

$$
\mathcal{F}_{\phi}
:
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\mapsto
\Delta\boldsymbol{\theta}^{(t)}.
$$

This learned update rule replaces a hand-designed optimizer step such as Adam, COBYLA, SPSA, or Nelder--Mead. The key idea is that the optimizer is no longer a fixed external routine; instead, it is a trainable component of the hybrid quantum--classical loop.

In [11]:
class TransformerQAOAOptimizer(nn.Module):
    def __init__(
        self,
        p: int,
        graph_feat_dim: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.p = p

        self.graph_proj = nn.Linear(graph_feat_dim, d_model)

        # Per-layer features:
        # gamma, beta, delta_gamma_prev, delta_beta_prev,
        # energy_now, energy_best, energy_gap, step_frac, remaining_frac
        self.layer_feat_proj = nn.Linear(9, d_model)
        self.layer_embed = nn.Embedding(p, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.out_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2),
        )

        self.log_step_scale = nn.Parameter(torch.tensor(-1.5, dtype=CFG.dtype))

    def forward(
        self,
        graph_feats: torch.Tensor,
        theta: torch.Tensor,
        delta_prev: torch.Tensor,
        energy_now: torch.Tensor,
        energy_best: torch.Tensor,
        step_idx: int,
        max_steps: int,
    ) -> torch.Tensor:
        batch_size = graph_feats.shape[0]
        p = self.p

        gammas = theta[:, :p]
        betas = theta[:, p:]
        dgammas = delta_prev[:, :p]
        dbetas = delta_prev[:, p:]

        energy_now_tok = energy_now.view(batch_size, 1, 1).expand(-1, p, -1)
        energy_best_tok = energy_best.view(batch_size, 1, 1).expand(-1, p, -1)
        energy_gap_tok = (energy_best - energy_now).view(batch_size, 1, 1).expand(-1, p, -1)

        step_frac_tok = torch.full(
            (batch_size, p, 1),
            fill_value=step_idx / max_steps,
            dtype=theta.dtype,
            device=theta.device,
        )
        remaining_frac_tok = torch.full(
            (batch_size, p, 1),
            fill_value=(max_steps - step_idx) / max_steps,
            dtype=theta.dtype,
            device=theta.device,
        )

        layer_feats = torch.cat(
            [
                gammas.unsqueeze(-1),
                betas.unsqueeze(-1),
                dgammas.unsqueeze(-1),
                dbetas.unsqueeze(-1),
                energy_now_tok,
                energy_best_tok,
                energy_gap_tok,
                step_frac_tok,
                remaining_frac_tok,
            ],
            dim=-1,
        )

        graph_token = self.graph_proj(graph_feats).unsqueeze(1)
        layer_tokens = self.layer_feat_proj(layer_feats)
        layer_tokens = layer_tokens + self.layer_embed.weight.unsqueeze(0)

        x = torch.cat([graph_token, layer_tokens], dim=1)
        h = self.encoder(x)
        layer_h = h[:, 1:, :]

        delta = self.out_head(layer_h)
        step_scale = torch.exp(self.log_step_scale)
        delta = step_scale * torch.tanh(delta)

        delta_gamma = delta[:, :, 0]
        delta_beta = delta[:, :, 1]
        return torch.cat([delta_gamma, delta_beta], dim=-1)

## Angle projection and batch utilities

QAOA angles have natural periodic or bounded domains. In this notebook, the cost angle is wrapped to the interval $[-\pi,\pi]$, while the mixer angle is clipped to the interval $[-\pi/2,\pi/2]$:

$$
\gamma_\ell
\leftarrow
\operatorname{wrap}_{[-\pi,\pi]}(\gamma_\ell),
\qquad
\beta_\ell
\leftarrow
\operatorname{clip}_{[-\pi/2,\pi/2]}(\beta_\ell).
$$

The wrapping operation maps any real-valued angle back into the periodic domain:

$$
\operatorname{wrap}_{[-\pi,\pi]}(x)
=
\left((x+\pi)\bmod 2\pi\right)-\pi.
$$

This is appropriate for the QAOA cost angle because the corresponding unitary evolution is periodic:

$$
e^{-i\gamma H_C}
\equiv
e^{-i(\gamma+2\pi)H_C},
$$

up to the relevant periodicity induced by the spectrum of $H_C$.

The clipping operation constrains the mixer angle to a bounded interval:

$$
\operatorname{clip}_{[-\pi/2,\pi/2]}(x)
=
\min
\left\{
\max
\left\{
x,
-\frac{\pi}{2}
\right\},
\frac{\pi}{2}
\right\}.
$$

Thus, the projected QAOA angles are

$$
\gamma_\ell^{(t+1)}
=
\operatorname{wrap}_{[-\pi,\pi]}
\left(
\gamma_\ell^{(t)}
+
\Delta\gamma_\ell^{(t)}
\right),
$$

and

$$
\beta_\ell^{(t+1)}
=
\operatorname{clip}_{[-\pi/2,\pi/2]}
\left(
\beta_\ell^{(t)}
+
\Delta\beta_\ell^{(t)}
\right).
$$

For a batch of $B$ graphs, the QAOA parameters can be represented as

$$
\boldsymbol{\Theta}^{(t)}
=
\left[
\boldsymbol{\theta}_1^{(t)},
\boldsymbol{\theta}_2^{(t)},
\ldots,
\boldsymbol{\theta}_B^{(t)}
\right]^\top
\in
\mathbb{R}^{B\times 2p}.
$$

The learned optimizer predicts a batch of updates

$$
\Delta\boldsymbol{\Theta}^{(t)}
=
\mathcal{F}_{\phi}
\left(
\mathcal{B}^{(t)}
\right)
\in
\mathbb{R}^{B\times 2p},
$$

where $\mathcal{B}^{(t)}$ denotes the batch of graph features, current angles, previous updates, and objective-history features.

The batch update is then

$$
\boldsymbol{\Theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\Theta}^{(t)}
+
\Delta\boldsymbol{\Theta}^{(t)}
\right),
$$

where $\Pi_{\Omega}$ applies the corresponding wrap and clip operations independently to every graph and every QAOA layer.

This projection stabilizes the learned optimizer and prevents the Transformer from spending modeling capacity on invalid or redundant update regions.

In [12]:
def project_angles(theta: torch.Tensor, p: int) -> torch.Tensor:
    gammas = theta[:, :p]
    betas = theta[:, p:]

    gammas = ((gammas + math.pi) % (2 * math.pi)) - math.pi
    betas = torch.clamp(betas, min=-math.pi / 2, max=math.pi / 2)

    return torch.cat([gammas, betas], dim=-1)


def random_angle_init(batch_size: int, p: int, device: str, dtype: torch.dtype, noise: float) -> torch.Tensor:
    gammas = (2.0 * torch.rand((batch_size, p), device=device, dtype=dtype) - 1.0) * noise
    betas = (2.0 * torch.rand((batch_size, p), device=device, dtype=dtype) - 1.0) * noise
    return torch.cat([gammas, betas], dim=-1)

In [13]:
def sample_batch(graphs: List[torch.Tensor], batch_size: int, device: str) -> List[torch.Tensor]:
    chosen = random.sample(graphs, batch_size)
    return [g.to(device) for g in chosen]


def batch_graph_features(graphs: List[torch.Tensor], device: str) -> torch.Tensor:
    feats = []
    for g in graphs:
        info = _graph_info(g)
        if info is not None:
            feats.append(info["features"])
        else:
            feats.append(_compute_upper_triangular_features(g.to(device)))
    return torch.stack(feats, dim=0).to(device)


def evaluate_batch_expectations(graphs: List[torch.Tensor], theta: torch.Tensor, p: int) -> torch.Tensor:
    if CFG.qc_simulator_backend == "cudaq":
        n_qubits = int(graphs[0].shape[0])
        edges_per_graph = tuple(_cudaq_edges(g) for g in graphs)
        if theta.requires_grad:
            return _CudaQQAOABatchedExpectation.apply(theta, n_qubits, p, edges_per_graph)
        theta_lists = theta.detach().to(dtype=torch.float64).cpu().tolist()
        values = _cudaq_observe_per_graph(n_qubits, p, edges_per_graph, theta_lists)
        return theta.new_tensor(values)

    vals = []
    for i, g in enumerate(graphs):
        gammas = theta[i, :p]
        betas = theta[i, p:]
        vals.append(qaoa_expectation(g, gammas, betas))
    return torch.stack(vals, dim=0)


def exact_batch_maxcuts(graphs: List[torch.Tensor], device: str) -> torch.Tensor:
    vals = []
    for g in graphs:
        info = _graph_info(g)
        if info is not None:
            vals.append(info["max_cut"])
        else:
            best, _ = maxcut_bruteforce(g.cpu())
            vals.append(best)
    return torch.tensor(vals, dtype=CFG.dtype, device=device)

## Meta-training objective

The learned optimizer is trained by unrolling $T$ update steps over graphs sampled from a training distribution $\mathcal{D}$. Each graph instance defines one QAOA optimization task.

For graph $G\sim\mathcal{D}$, the approximation ratio at step $t$ is

$$
\rho_t(G)
=
\frac{
f_G(\boldsymbol{\theta}^{(t)})
}{
C_G^\star
},
$$

where $C_G^\star$ is the exact MaxCut value and $f_G(\boldsymbol{\theta}^{(t)})$ is the expected QAOA cut value at step $t$.

The best-so-far approximation ratio over the unrolled trajectory is

$$
\rho_{\mathrm{best}}(G)
=
\max_{0\leq \tau \leq T}
\rho_\tau(G).
$$

The implementation uses a trajectory-shaped loss of the form

$$
\mathcal{L}(G;\phi)
=
-
a_1
\frac{1}{T}
\sum_{t=0}^{T-1}
\rho_t(G)
-
a_2
\rho_T(G)
-
a_3
\rho_{\mathrm{best}}(G)
-
a_4
\frac{1}{T}
\sum_{t=1}^{T-1}
\frac{
f_G(\boldsymbol{\theta}^{(t)})
-
f_G(\boldsymbol{\theta}^{(t-1)})
}{
C_G^\star
}
+
\lambda_\Delta
\sum_{t=0}^{T-1}
\left\|
\Delta\boldsymbol{\theta}^{(t)}
\right\|_2^2 .
$$

Here, $a_1,a_2,a_3,a_4\geq 0$ control the relative importance of trajectory quality, final-step quality, best-so-far quality, and improvement between consecutive steps. The coefficient $\lambda_\Delta\geq 0$ regularizes the update magnitude.

The first term encourages the optimizer to maintain good average performance throughout the trajectory:

$$
-
\frac{1}{T}
\sum_{t=0}^{T-1}
\rho_t(G).
$$

The second term encourages a high-quality final QAOA parameter setting:

$$
-
\rho_T(G).
$$

The third term rewards the best solution discovered anywhere along the trajectory:

$$
-
\rho_{\mathrm{best}}(G).
$$

The fourth term encourages positive progress between consecutive optimization steps:

$$
-
\frac{1}{T}
\sum_{t=1}^{T-1}
\frac{
f_G(\boldsymbol{\theta}^{(t)})
-
f_G(\boldsymbol{\theta}^{(t-1)})
}{
C_G^\star
}.
$$

The final term penalizes overly large learned updates:

$$
\lambda_\Delta
\sum_{t=0}^{T-1}
\left\|
\Delta\boldsymbol{\theta}^{(t)}
\right\|_2^2 .
$$

The meta-training problem is therefore

$$
\phi^\star
=
\arg\min_{\phi}
\mathbb{E}_{G\sim\mathcal{D}}
\left[
\mathcal{L}(G;\phi)
\right].
$$

For a minibatch of $B$ graphs, this expectation is approximated by

$$
\widehat{\mathcal{L}}(\phi)
=
\frac{1}{B}
\sum_{b=1}^{B}
\mathcal{L}(G_b;\phi),
\qquad
G_b\sim\mathcal{D}.
$$

This objective encourages good final quality, strong best-so-far quality, stable intermediate progress, and controlled update magnitudes. In an SC tutorial setting, the same unrolled structure is useful for discussing GPU occupancy, batched objective evaluation, asynchronous quantum-expectation calls, and amortized optimization across many related graph instances.

In [14]:
@torch.no_grad()
def evaluate_policy_mean_ratio(model: nn.Module, graphs: List[torch.Tensor], cfg: Config) -> float:
    model.eval()

    batch_graphs = [g.to(cfg.device) for g in graphs]
    graph_feats = batch_graph_features(batch_graphs, cfg.device)
    maxcuts = exact_batch_maxcuts(batch_graphs, cfg.device)

    theta = torch.zeros((len(batch_graphs), 2 * cfg.p), dtype=cfg.dtype, device=cfg.device)
    delta_prev = torch.zeros_like(theta)
    energy_best = torch.full((len(batch_graphs),), -1e9, dtype=cfg.dtype, device=cfg.device)

    for t in range(cfg.unroll_steps):
        energy_now = evaluate_batch_expectations(batch_graphs, theta, cfg.p)
        energy_best = torch.maximum(energy_best, energy_now)

        delta = model(
            graph_feats=graph_feats,
            theta=theta,
            delta_prev=delta_prev,
            energy_now=energy_now,
            energy_best=energy_best,
            step_idx=t,
            max_steps=cfg.unroll_steps,
        )
        theta = project_angles(theta + delta, cfg.p)
        delta_prev = delta

    final_energy = evaluate_batch_expectations(batch_graphs, theta, cfg.p)
    best_energy = torch.maximum(energy_best, final_energy)
    return float((best_energy / maxcuts).mean().item())


def meta_train(
    model: nn.Module,
    train_graphs: List[torch.Tensor],
    val_graphs: List[torch.Tensor],
    cfg: Config,
) -> Dict[str, torch.Tensor]:
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    best_val_ratio = -1.0
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(1, cfg.meta_epochs + 1):
        batch_graphs = sample_batch(train_graphs, cfg.batch_size, cfg.device)
        graph_feats = batch_graph_features(batch_graphs, cfg.device)
        maxcuts = exact_batch_maxcuts(batch_graphs, cfg.device)

        theta = random_angle_init(
            batch_size=cfg.batch_size,
            p=cfg.p,
            device=cfg.device,
            dtype=cfg.dtype,
            noise=cfg.init_angle_noise,
        )
        delta_prev = torch.zeros_like(theta)
        energy_best = torch.full((cfg.batch_size,), -1e9, dtype=cfg.dtype, device=cfg.device)

        ratio_losses = []
        improve_losses = []
        prev_energy = None

        for t in range(cfg.unroll_steps):
            energy_now = evaluate_batch_expectations(batch_graphs, theta, cfg.p)
            energy_best = torch.maximum(energy_best, energy_now)

            delta = model(
                graph_feats=graph_feats,
                theta=theta,
                delta_prev=delta_prev,
                energy_now=energy_now,
                energy_best=energy_best,
                step_idx=t,
                max_steps=cfg.unroll_steps,
            )
            theta = project_angles(theta + delta, cfg.p)
            delta_prev = delta

            ratio_now = energy_now / maxcuts
            ratio_losses.append(-ratio_now.mean())

            if prev_energy is None:
                improve_losses.append(torch.zeros((), dtype=cfg.dtype, device=cfg.device))
            else:
                improve_losses.append(-((energy_now - prev_energy) / maxcuts).mean())

            prev_energy = energy_now

        final_energy = evaluate_batch_expectations(batch_graphs, theta, cfg.p)
        best_energy = torch.maximum(energy_best, final_energy)

        final_ratio = final_energy / maxcuts
        best_ratio = best_energy / maxcuts
        step_penalty = 0.01 * delta_prev.pow(2).mean()

        loss = (
            0.20 * torch.stack(ratio_losses).mean()
            + 0.10 * torch.stack(improve_losses).mean()
            - 0.25 * final_ratio.mean()
            - 0.65 * best_ratio.mean()
            + step_penalty
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if epoch == 1 or epoch % 20 == 0 or epoch == cfg.meta_epochs:
            val_ratio = evaluate_policy_mean_ratio(model, val_graphs, cfg)
            if val_ratio > best_val_ratio:
                best_val_ratio = val_ratio
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            print(
                f"[meta-train] epoch={epoch:03d} | "
                f"loss={loss.item():+.4f} | "
                f"train_final_ratio={final_ratio.mean().item():.4f} | "
                f"train_best_ratio={best_ratio.mean().item():.4f} | "
                f"val_best_ratio={val_ratio:.4f} | "
                f"step_scale={torch.exp(model.log_step_scale).item():.4f}"
            )

            if cfg.early_stop_patience > 0 and patience_counter >= cfg.early_stop_patience:
                print(
                    f"[meta-train] early stop at epoch={epoch}: "
                    f"no val improvement for {patience_counter} checks "
                    f"(best_val_best_ratio={best_val_ratio:.4f})"
                )
                break

    return best_state

## Inference-time refinement and reporting

After the Transformer produces a good parameter trajectory, the notebook optionally applies a short local refinement step using Adam. This hybrid policy is useful pedagogically because it separates global learned initialization from local continuous optimization:

$$
\boldsymbol{\theta}^{(0)}
\xrightarrow{\text{Transformer policy}}
\boldsymbol{\theta}_{\mathrm{learned}}
\xrightarrow{\text{local refinement}}
\boldsymbol{\theta}_{\mathrm{final}}.
$$

The Transformer policy performs amortized optimization:

$$
\boldsymbol{\theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\theta}^{(t)}
+
\mathcal{F}_{\phi}
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\right),
\qquad
t=0,\ldots,T-1.
$$

The learned output after $T$ steps is

$$
\boldsymbol{\theta}_{\mathrm{learned}}
=
\boldsymbol{\theta}^{(T)}.
$$

The optional local refinement then initializes Adam at $\boldsymbol{\theta}_{\mathrm{learned}}$ and performs a small number of gradient-based updates:

$$
\boldsymbol{\theta}_{\mathrm{final}}
=
\operatorname{AdamRefine}
\left(
\boldsymbol{\theta}_{\mathrm{learned}},
G
\right).
$$

Equivalently, the refinement stage solves the local problem

$$
\boldsymbol{\theta}_{\mathrm{final}}
\approx
\arg\max_{\boldsymbol{\theta}\in\Omega}
f_G(\boldsymbol{\theta})
\quad
\text{initialized at}
\quad
\boldsymbol{\theta}_{\mathrm{learned}}.
$$

The reported quantities are:

- best Transformer-only expected cut,
- refined expected cut,
- approximation ratio,
- final QAOA angles,
- exact MaxCut value,
- most-likely bitstring under the final state.

The best Transformer-only expected cut is

$$
f_{\mathrm{learned,best}}
=
\max_{0\leq t\leq T}
f_G(\boldsymbol{\theta}^{(t)}).
$$

The refined expected cut is

$$
f_{\mathrm{refined}}
=
f_G(\boldsymbol{\theta}_{\mathrm{final}}).
$$

The final approximation ratio is

$$
\rho_{\mathrm{final}}
=
\frac{
f_G(\boldsymbol{\theta}_{\mathrm{final}})
}{
C_G^\star
}.
$$

The final QAOA angles are

$$
\boldsymbol{\theta}_{\mathrm{final}}
=
[
\gamma_1,\ldots,\gamma_p,
\beta_1,\ldots,\beta_p
].
$$

The exact MaxCut value is

$$
C_G^\star
=
\max_{\mathbf{z}\in\{0,1\}^n}
C_G(\mathbf{z}).
$$

The most-likely bitstring under the final QAOA state is

$$
\mathbf{z}_{\mathrm{ML}}
=
\arg\max_{\mathbf{z}\in\{0,1\}^n}
\left|
\langle
\mathbf{z}
|
\psi_p(\boldsymbol{\theta}_{\mathrm{final}};G)
\rangle
\right|^2.
$$

Its corresponding cut value is

$$
C_G(\mathbf{z}_{\mathrm{ML}})
=
\sum_{(i,j)\in E}
\mathbf{1}
[
z_i\neq z_j
].
$$

This reporting structure distinguishes three complementary outcomes: the quality of the learned optimizer alone, the benefit of local refinement, and the quality of the final sampled solution.

In [15]:
def local_refine_theta(
    adj: torch.Tensor,
    theta_init: torch.Tensor,
    cfg: Config,
) -> Tuple[torch.Tensor, float]:
    theta = theta_init.detach().clone().to(cfg.device).to(cfg.dtype)
    theta.requires_grad_(True)

    optimizer = torch.optim.Adam([theta], lr=cfg.refine_lr)

    best_theta = theta.detach().clone()
    best_energy = float(qaoa_expectation(adj, theta[:cfg.p], theta[cfg.p:]).detach().item())

    for _ in range(cfg.refine_steps):
        energy = qaoa_expectation(adj, theta[:cfg.p], theta[cfg.p:])
        loss = -energy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            theta[:] = project_angles(theta.unsqueeze(0), cfg.p).squeeze(0)
            current_energy = float(qaoa_expectation(adj, theta[:cfg.p], theta[cfg.p:]).item())
            if current_energy > best_energy:
                best_energy = current_energy
                best_theta = theta.detach().clone()

    return best_theta, best_energy


def run_learned_optimizer_on_graph(
    model: nn.Module,
    adj: torch.Tensor,
    cfg: Config,
) -> Dict[str, Any]:
    model.eval()
    adj = adj.to(cfg.device)

    graph_feats = upper_triangular_features(adj).unsqueeze(0).to(cfg.device)

    theta = torch.zeros((1, 2 * cfg.p), dtype=cfg.dtype, device=cfg.device)
    delta_prev = torch.zeros_like(theta)
    energy_best = torch.full((1,), -1e9, dtype=cfg.dtype, device=cfg.device)

    best_theta = theta.clone()
    history = []

    with torch.no_grad():
        for t in range(cfg.unroll_steps):
            energy_now = qaoa_expectation(adj, theta[0, :cfg.p], theta[0, cfg.p:]).unsqueeze(0)
            if float(energy_now.item()) > float(energy_best.item()):
                best_theta = theta.clone()

            energy_best = torch.maximum(energy_best, energy_now)

            history.append(
                {
                    "step": t,
                    "energy": float(energy_now.item()),
                    "gammas": theta[0, :cfg.p].detach().cpu().tolist(),
                    "betas": theta[0, cfg.p:].detach().cpu().tolist(),
                }
            )

            delta = model(
                graph_feats=graph_feats,
                theta=theta,
                delta_prev=delta_prev,
                energy_now=energy_now,
                energy_best=energy_best,
                step_idx=t,
                max_steps=cfg.unroll_steps,
            )
            theta = project_angles(theta + delta, cfg.p)
            delta_prev = delta

        final_energy_raw = float(qaoa_expectation(adj, theta[0, :cfg.p], theta[0, cfg.p:]).item())
        best_energy_raw = float(qaoa_expectation(adj, best_theta[0, :cfg.p], best_theta[0, cfg.p:]).item())

        if final_energy_raw > best_energy_raw:
            best_theta = theta.clone()
            best_energy_raw = final_energy_raw

    refined_theta, refined_energy = local_refine_theta(
        adj=adj,
        theta_init=best_theta.squeeze(0),
        cfg=cfg,
    )

    best_cut, best_bitstring = maxcut_bruteforce(adj)
    most_likely_z, most_likely_cut = most_likely_bitstring(adj, refined_theta[:cfg.p], refined_theta[cfg.p:])

    return {
        "history": history,
        "transformer_best_energy": best_energy_raw,
        "refined_energy": refined_energy,
        "final_gammas": refined_theta[:cfg.p].detach().cpu().tolist(),
        "final_betas": refined_theta[cfg.p:].detach().cpu().tolist(),
        "exact_maxcut": best_cut,
        "exact_best_bitstring": best_bitstring,
        "approx_ratio": refined_energy / best_cut,
        "most_likely_bitstring": most_likely_z,
        "most_likely_cut": most_likely_cut,
    }


def print_demo_result(adj: torch.Tensor, result: dict) -> None:
    print("\n" + "=" * 72)
    print(f"{adj.shape[0]}-QUBIT MAXCUT DEMO (IMPROVED)")
    print("=" * 72)
    print(f"Edges: {edge_list_from_adj(adj.cpu())}")
    print(f"Exact MaxCut value: {result['exact_maxcut']:.1f}")
    print(f"One optimal bitstring: {result['exact_best_bitstring']}")
    print("-" * 72)
    print("Optimization trajectory:")
    for item in result["history"]:
        gammas_str = ", ".join(f"{x:+.4f}" for x in item["gammas"])
        betas_str = ", ".join(f"{x:+.4f}" for x in item["betas"])
        print(
            f"step={item['step']:02d} | "
            f"E[C]={item['energy']:.4f} | "
            f"gammas=[{gammas_str}] | "
            f"betas=[{betas_str}]"
        )
    print("-" * 72)
    print(f"Best Transformer-only expected cut: {result['transformer_best_energy']:.4f}")
    print(f"Refined expected cut: {result['refined_energy']:.4f}")
    print(f"Approximation ratio: {result['approx_ratio']:.4f}")
    print(f"Final gammas: {['{:+.4f}'.format(x) for x in result['final_gammas']]}")
    print(f"Final betas:  {['{:+.4f}'.format(x) for x in result['final_betas']]}")
    print(f"Most likely sampled bitstring: {result['most_likely_bitstring']}")
    print(f"Cut value of most likely bitstring: {result['most_likely_cut']:.1f}")
    print("=" * 72)

## Build the hands-on modules

This cell constructs the complete all the hands-on modules, including the demonstration graph, training graphs, validation graphs, cached graph invariants, and Transformer optimizer model.

For the default live-demo setup, the notebook uses a small configuration:

$$
n=5,
\qquad
p=2,
\qquad
|\mathcal{D}_{\mathrm{train}}|=16,
\qquad
|\mathcal{D}_{\mathrm{val}}|=4.
$$

Here, $n$ is the number of graph vertices, equivalently the number of qubits in the QAOA circuit, and $p$ is the QAOA circuit depth. The training and validation graph sets are

$$
\mathcal{D}_{\mathrm{train}}
=
\left\{
G_1^{\mathrm{train}},
G_2^{\mathrm{train}},
\ldots,
G_{16}^{\mathrm{train}}
\right\},
$$

and

$$
\mathcal{D}_{\mathrm{val}}
=
\left\{
G_1^{\mathrm{val}},
G_2^{\mathrm{val}},
\ldots,
G_{4}^{\mathrm{val}}
\right\}.
$$

For each graph $G$, the notebook precomputes and caches graph-invariant quantities:

$$
\mathcal{C}(G)
=
\left\{
E,\,
\mathbf{x}_G,\,
\operatorname{diag}(H_C),\,
C_G^\star
\right\}.
$$

Here, $E$ is the edge list, $\mathbf{x}_G$ is the graph feature vector, $\operatorname{diag}(H_C)$ is the diagonal of the MaxCut cost Hamiltonian in the computational basis, and $C_G^\star$ is the exact MaxCut value.

The Transformer optimizer is then initialized as a learned update rule

$$
\mathcal{F}_{\phi}
:
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\mapsto
\Delta\boldsymbol{\theta}^{(t)}.
$$

During training, this learned optimizer is unrolled over each graph instance:

$$
\boldsymbol{\theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\theta}^{(t)}
+
\mathcal{F}_{\phi}
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\right).
$$

The default live-demo configuration is intentionally small so that the full workflow can run interactively during a tutorial session. It is designed to demonstrate the core ideas clearly:

- graph construction,
- graph-invariant caching,
- QAOA expectation evaluation,
- Transformer-based learned updates,
- unrolled meta-training,
- inference-time reporting.

For review-scale experiments, set

```python
SC_TUTORIAL_FAST_MODE = False

In [16]:
print("Configuration:")
for k, v in CFG.__dict__.items():
    print(f"  {k}: {v}")

demo_edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]
if CFG.n_qubits == 5:
    demo_adj = make_adj(CFG.n_qubits, demo_edges, device=CFG.device, dtype=CFG.dtype)
else:
    demo_adj = random_connected_graph(CFG.n_qubits, p_edge=0.3).to(
        device=CFG.device,
        dtype=CFG.dtype,
    )

demo_key = graph_key(demo_adj.cpu())

train_graphs = build_unique_graphs(
    num_graphs=CFG.train_graph_count,
    n_qubits=CFG.n_qubits,
    excluded_keys={demo_key},
)
train_keys = {graph_key(g) for g in train_graphs}

val_graphs = build_unique_graphs(
    num_graphs=CFG.val_graph_count,
    n_qubits=CFG.n_qubits,
    excluded_keys=train_keys.union({demo_key}),
)

# Move all graphs to the target device once and cache invariants.
train_graphs = precompute_graph_info(train_graphs, CFG.device)
val_graphs = precompute_graph_info(val_graphs, CFG.device)
demo_adj = precompute_graph_info([demo_adj], CFG.device)[0]

graph_feat_dim = upper_triangular_features(train_graphs[0]).numel()

model = TransformerQAOAOptimizer(
    p=CFG.p,
    graph_feat_dim=graph_feat_dim,
    d_model=CFG.d_model,
    nhead=CFG.nhead,
    num_layers=CFG.num_layers,
    dim_feedforward=CFG.dim_feedforward,
    dropout=CFG.dropout,
).to(device=CFG.device, dtype=CFG.dtype)

print(f"\nTraining graphs: {len(train_graphs)}")
print(f"Validation graphs: {len(val_graphs)}")
print(f"Graph feature dimension: {graph_feat_dim}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Configuration:
  n_qubits: 5
  p: 2
  train_graph_count: 16
  val_graph_count: 4
  batch_size: 2
  meta_epochs: 5
  unroll_steps: 3
  lr: 0.002
  d_model: 32
  nhead: 4
  num_layers: 1
  dim_feedforward: 64
  dropout: 0.0
  weight_decay: 1e-05
  init_angle_noise: 0.2
  refine_steps: 5
  refine_lr: 0.05
  seed: 42
  device: cuda
  dtype: torch.float32
  complex_dtype: torch.complex64
  qc_simulator_backend: cudaq
  cudaq_target: nvidia
  cudaq_target_option: fp32
  cudaq_grad_epsilon: 0.0001
  cudaq_kernel_mode: jit
  cudaq_use_async_forward: True
  early_stop_patience: 5



Training graphs: 16
Validation graphs: 4
Graph feature dimension: 16
Model parameters: 10,659


/tmp/ipykernel_7030/2208588439.py:32: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


## Train the intrinsic optimizer

This cell performs the meta-training loop. During training, the Transformer learns an update policy

$$
\mathcal{F}_{\phi}
:
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\mapsto
\Delta\boldsymbol{\theta}^{(t)}.
$$

For each graph $G\sim\mathcal{D}_{\mathrm{train}}$, the learned optimizer is unrolled for $T$ update steps:

$$
\boldsymbol{\theta}^{(t+1)}
=
\Pi_{\Omega}
\left(
\boldsymbol{\theta}^{(t)}
+
\mathcal{F}_{\phi}
\left(
G,
\boldsymbol{\theta}^{(t)},
\Delta\boldsymbol{\theta}^{(t-1)},
E_t,
E_t^{\mathrm{best}},
t/T
\right)
\right),
\qquad
t=0,\ldots,T-1.
$$

The training objective is computed from the full unrolled trajectory. A typical trajectory-shaped loss is

$$
\mathcal{L}(G;\phi)
=
-
a_1
\frac{1}{T}
\sum_{t=0}^{T-1}
\rho_t(G)
-
a_2
\rho_T(G)
-
a_3
\rho_{\mathrm{best}}(G)
+
\lambda_\Delta
\sum_{t=0}^{T-1}
\left\|
\Delta\boldsymbol{\theta}^{(t)}
\right\|_2^2,
$$

where

$$
\rho_t(G)
=
\frac{
f_G(\boldsymbol{\theta}^{(t)})
}{
C_G^\star
},
\qquad
\rho_{\mathrm{best}}(G)
=
\max_{0\leq \tau\leq T}
\rho_\tau(G).
$$

For a minibatch of $B$ training graphs, the empirical meta-training loss is

$$
\widehat{\mathcal{L}}_{\mathrm{train}}(\phi)
=
\frac{1}{B}
\sum_{b=1}^{B}
\mathcal{L}(G_b;\phi),
\qquad
G_b\sim\mathcal{D}_{\mathrm{train}}.
$$

The Transformer parameters are updated by gradient-based optimization:

$$
\phi
\leftarrow
\phi
-
\eta
\nabla_{\phi}
\widehat{\mathcal{L}}_{\mathrm{train}}(\phi),
$$

where $\eta$ is the learning rate.

After each validation interval, the learned optimizer is evaluated on held-out graphs

$$
G\sim\mathcal{D}_{\mathrm{val}}.
$$

The validation metric is the mean approximation ratio

$$
\overline{\rho}_{\mathrm{val}}
=
\frac{1}{|\mathcal{D}_{\mathrm{val}}|}
\sum_{G\in\mathcal{D}_{\mathrm{val}}}
\rho_{\mathrm{best}}(G).
$$

The best checkpoint is selected according to validation approximation ratio:

$$
\phi_{\mathrm{best}}
=
\arg\max_{\phi}
\overline{\rho}_{\mathrm{val}}(\phi).
$$

This training procedure teaches the Transformer to act as an intrinsic optimizer: it observes the current QAOA optimization state and predicts the next parameter update, rather than relying on a fixed external optimizer such as Adam, COBYLA, SPSA, or Nelder--Mead.

In [17]:
best_state = meta_train(model, train_graphs, val_graphs, CFG)
model.load_state_dict(best_state)
print("Loaded the best validation checkpoint.")

[meta-train] epoch=001 | loss=-0.5420 | train_final_ratio=0.4106 | train_best_ratio=0.5265 | val_best_ratio=0.5813 | step_scale=0.2227


[meta-train] epoch=005 | loss=-0.8840 | train_final_ratio=0.8213 | train_best_ratio=0.8213 | val_best_ratio=0.7634 | step_scale=0.2231
Loaded the best validation checkpoint.


## Run the learned optimizer on the held-out demonstration graph

Let's wrap up — this final demonstration executes the learned optimizer, then applies local refinement:

In [18]:
result = run_learned_optimizer_on_graph(model, demo_adj, CFG)
print_demo_result(demo_adj.cpu(), result)


5-QUBIT MAXCUT DEMO (IMPROVED)
Edges: [(0, 1), (0, 2), (0, 4), (1, 2), (2, 3), (3, 4)]
Exact MaxCut value: 5.0
One optimal bitstring: 10010
------------------------------------------------------------------------
Optimization trajectory:
step=00 | E[C]=3.0000 | gammas=[+0.0000, +0.0000] | betas=[+0.0000, +0.0000]
step=01 | E[C]=3.1920 | gammas=[+0.0098, -0.1521] | betas=[+0.0217, -0.1185]
step=02 | E[C]=3.6240 | gammas=[+0.0178, -0.3034] | betas=[+0.0458, -0.2349]
------------------------------------------------------------------------
Best Transformer-only expected cut: 3.9782
Refined expected cut: 4.0872
Approximation ratio: 0.8174
Final gammas: ['-0.1041', '-0.6237']
Final betas:  ['+0.0179', '-0.3504']
Most likely sampled bitstring: 10010
Cut value of most likely bitstring: 5.0


---
## Coming up next — Notebook 02

The morning block established the programming model and one complete hybrid workflow. The afternoon block scales up to three further QML model families, each of which stresses the simulation stack in a different way:

| Notebook | What it does |
|:---|:---|
| `02_qfwp.ipynb` | **Quantum fast weights:** a compact quantum circuit emits the weight updates that reprogram a classical network, replacing explicit recurrence. |
| `03_qkan_basics.ipynb` | **Quantum-inspired** Kolmogorov–Arnold Networks (QKAN) as a tensor-network backbone for parameter-efficient LLMs. |
| `04_cutn-qsvm.ipynb` | **Quantum-enhanced** SVM: classical SVM with a quantum kernel, scaled via cuQuantum batched contraction (and an optional multi-stream cuTENSOR backend). |

Enjoy the break — see you in the afternoon block. 🚀